# Система Skills — вариативные мини-промпты

**Skill** — небольшой текстовый блок, который добавляется в системный промпт динамически,  
расширяя поведение агента под конкретный тип запроса.

## Принцип работы

```
Базовый промпт  (присутствует всегда)
  + Skill A     (подключается если нужен)
  + Skill B     (подключается если нужен)
       ↓
  Итоговый системный промпт → LLM
```

## Преимущества перед монолитным промптом


In [ ]:
%pip install -q langchain langchain-groq python-dotenv

## Настройка

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage

load_dotenv()  # читает GROQ_API_KEY из .env

llm = ChatGroq(model="llama3-8b-8192", temperature=0)

## 1. Библиотека Skills

Каждый skill — строка с инструкциями. Ключ словаря — имя skill, которое используется для активации.

In [ ]:
SKILLS: dict[str, str] = {
    "detailed": """[Skill: Детальный ответ]
Давай развёрнутые ответы с примерами и пояснениями.
Структурируй информацию по пунктам. Не сокращай без необходимости.""",

    "concise": """[Skill: Краткий ответ]
Отвечай максимально лаконично: не более 2–3 предложений.
Только суть — без вводных слов, примеров и повторений.""",

    "formal": """[Skill: Формальный стиль]
Используй деловой стиль общения.
Обращайся на «Вы», избегай разговорных выражений и сокращений.""",

    "step_by_step": """[Skill: Пошаговые инструкции]
Разбивай ответ на нумерованные шаги.
Каждый шаг — одно конкретное действие. Добавляй промежуточные проверки.""",

    "critical": """[Skill: Критический анализ]
Указывай не только плюсы, но и минусы, ограничения и риски.
Не давай однозначных оценок без оговорок.""",
}

BASE_PROMPT = """Ты — универсальный AI-ассистент.
Отвечаешь на вопросы чётко и по делу.
Если вопрос непонятен — уточни, что именно имеется в виду.
Отвечай на русском языке."""

print(f"Доступные skills: {list(SKILLS.keys())}")

## 2. Автоопределение Skills по запросу

Простейший вариант — ключевые слова. В production можно использовать отдельный LLM-классификатор.

In [ ]:
SKILL_KEYWORDS: dict[str, list[str]] = {
    "concise":     ["кратко", "коротко", "быстро", "вкратце", "одним словом"],
    "detailed":    ["подробно", "детально", "расскажи", "объясни", "развёрнуто"],
    "formal":      ["официально", "формально", "деловой стиль", "для документа"],
    "step_by_step":["шаги", "пошагово", "как сделать", "инструкция", "порядок"],
    "critical":    ["минусы", "риски", "ограничения", "недостатки", "критически"],
}


def detect_skills(query: str) -> list[str]:
    """Определяет подходящие Skills по ключевым словам в запросе."""
    query_lower = query.lower()
    return [
        skill
        for skill, keywords in SKILL_KEYWORDS.items()
        if any(kw in query_lower for kw in keywords)
    ]


# Демо автоопределения
test_queries = [
    "Объясни подробно, что такое Docker",
    "Как установить Python? Пошагово",
    "Кратко: что такое REST API?",
    "Какие риски у микросервисной архитектуры?",
    "Привет!",
]

for q in test_queries:
    skills = detect_skills(q)
    print(f"{q[:45]:<45} → {skills or ['(базовый)']})")

## 3. Сборка системного промпта

In [ ]:
def build_system_prompt(skills: list[str]) -> str:
    """Собирает итоговый промпт: базовый + выбранные Skills."""
    parts = [BASE_PROMPT]
    for name in skills:
        if name in SKILLS:
            parts.append(SKILLS[name])
    return "\n\n".join(parts)


# Пример: посмотрим на промпт с двумя Skills
prompt = build_system_prompt(["concise", "critical"])
print(prompt)

## 4. Запрос с динамическим промптом

In [ ]:
def ask(query: str, force_skills: list[str] | None = None) -> str:
    """
    Отправляет запрос к LLM, подключая нужные Skills.

    Args:
        query: вопрос пользователя
        force_skills: явно задать список Skills (иначе — автоопределение)
    """
    active_skills = force_skills if force_skills is not None else detect_skills(query)
    system_prompt = build_system_prompt(active_skills)

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query),
    ])

    print(f"[Skills: {active_skills or 'базовый'}]")
    return response.content

## 5. Демо — один вопрос, разные Skills

In [ ]:
question = "Что такое микросервисная архитектура?"

print("=" * 60)
print("КРАТКО:")
print(ask(question, force_skills=["concise"]))

print()
print("=" * 60)
print("ПОДРОБНО + КРИТИЧЕСКИ:")
print(ask(question, force_skills=["detailed", "critical"]))

In [ ]:
# Автоопределение Skills из текста запроса
print(ask("Как установить и настроить PostgreSQL? Пошагово, пожалуйста"))

## 6. Расширение: добавить новый Skill

Достаточно добавить одну запись в словарь — без изменения логики агента.

In [ ]:
# Добавляем новый skill прямо во время работы
SKILLS["emoji"] = """[Skill: Emoji-стиль]
Добавляй релевантные эмодзи в ответ для наглядности.
Используй не более 1 эмодзи на предложение."""

SKILL_KEYWORDS["emoji"] = ["с эмодзи", "весело", "неформально"]

print(ask("Кратко объясни, что такое API, с эмодзи"))